In [1]:
import os
print(sorted(os.listdir("../results/tables")))


['.DS_Store', 'distilbert_v2_val_by_source.csv', 'distilbert_v2_val_probs.csv', 'lora_val_probs_epoch2.csv', 'qlora_val_probs_epoch1.csv', 'tfidf_lr_v2_val_probs.csv']


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score

T = "../results/tables/"
files = {
    "TF-IDF + LR": "tfidf_lr_v2_val_probs.csv",
    "DistilBERT": "distilbert_v2_val_probs.csv",
    "Qwen LoRA": "lora_val_probs_epoch2.csv",
    "Qwen QLoRA": "qlora_val_probs_epoch1.csv",
}
frames = {name: pd.read_csv(T + f).set_index("id") for name, f in files.items()}

ids = frames["DistilBERT"].index
y = frames["DistilBERT"]["y"].values
probs = {}
for name, f in frames.items():
    assert set(f.index) == set(ids), name + ": different rows from the others"
    assert (f.loc[ids, "y"].values == y).all(), name + ": labels differ"
    probs[name] = f.loc[ids, "prob_attack"].values

print("validation rows:", len(ids), "| attacks:", int(y.sum()), "| safe:", int((y == 0).sum()))

validation rows: 3298 | attacks: 669 | safe: 2629


In [3]:
def recall_at_fp(y, p, rate):
    cut = np.quantile(p[y == 0], 1 - rate)
    return float((p[y == 1] > cut).mean())


rows = []
for name, p in probs.items():
    pred = (p >= 0.5).astype(int)
    rows.append({
        "model": name,
        "AP": average_precision_score(y, p),
        "recall@1%FP": recall_at_fp(y, p, 0.01),
        "recall@0.5%FP": recall_at_fp(y, p, 0.005),
        "recall@0.2%FP": recall_at_fp(y, p, 0.002),
        "precision@0.5": precision_score(y, pred, zero_division=0),
        "recall@0.5": recall_score(y, pred, zero_division=0),
        "safe flagged@0.5": int(((y == 0) & (pred == 1)).sum()),
    })
comparison = pd.DataFrame(rows).round(3)
comparison.to_csv(T + "validation_comparison.csv", index=False)
print(comparison.to_string(index=False))

      model    AP  recall@1%FP  recall@0.5%FP  recall@0.2%FP  precision@0.5  recall@0.5  safe flagged@0.5
TF-IDF + LR 0.953        0.815          0.786          0.631          0.979       0.773                11
 DistilBERT 0.966        0.846          0.752          0.692          0.929       0.894                46
  Qwen LoRA 0.973        0.859          0.789          0.744          0.955       0.862                27
 Qwen QLoRA 0.972        0.849          0.825          0.759          0.938       0.897                40


In [4]:
rows = []
for name, p in probs.items():
    cut = float(np.quantile(p[y == 0], 0.99))
    pred = (p > cut).astype(int)
    rows.append({
        "model": name,
        "threshold_1pct_fp": round(cut, 4),
        "val_recall": round(float(recall_score(y, pred)), 3),
        "val_precision": round(float(precision_score(y, pred, zero_division=0)), 3),
        "val_safe_flagged": int(((y == 0) & (pred == 1)).sum()),
    })
thresholds = pd.DataFrame(rows)
thresholds.to_csv(T + "thresholds_from_validation.csv", index=False)
print(thresholds.to_string(index=False))

      model  threshold_1pct_fp  val_recall  val_precision  val_safe_flagged
TF-IDF + LR             0.3548       0.815          0.953                27
 DistilBERT             0.7821       0.846          0.954                27
  Qwen LoRA             0.5115       0.859          0.955                27
 Qwen QLoRA             0.7834       0.849          0.955                27
